# NutriScan — AI Food Detection & Classification
**Capstone · Deakin Master of Applied AI (Bridge) · Mohammad Alrashed**

NutriScan wants users to photograph a meal and instantly get the food items, where they are on the plate, and their nutrition.
The brief asks for two models:

1. **Object detection** to locate food items in an image.
2. **Classification** to recognise the food type.

This notebook goes end to end: data audit → EDA → stratified split → two classifiers compared → detector training → a combined
detect-then-classify pipeline → error analysis → nutrition mapping → business insights → export → a live Streamlit app.

| Section | What happens |
|---|---|
| 0 | Setup, paths, reproducibility |
| 1 | Data loading and integrity checks |
| 2 | Exploratory data analysis |
| 3 | Stratified train / val / test split |
| 4 | Classification: YOLO11s-cls vs EfficientNet-B0 |
| 5 | Detection: auto-generated boxes (Grounding DINO) → YOLO11s detector |
| 6 | Two-stage pipeline (detect → classify) and error analysis |
| 7 | Nutrition mapping and meal analytics |
| 8 | Business insights, limitations, next steps |
| 9 | Export artefacts and launch the Streamlit app |

## 0. Setup

In [ ]:
import os, sys, subprocess, importlib
QUICK = os.environ.get("NS_QUICK") == "1"          # tiny smoke-test mode (CPU)
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
def pip(*pk):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pk], check=False)
for mod, pkg in [("ultralytics", "ultralytics"), ("streamlit", "streamlit"), ("plotly", "plotly")]:
    if importlib.util.find_spec(mod) is None:
        pip(pkg)
if IN_COLAB and not QUICK and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import glob, json, random, shutil, time, zipfile, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image, ImageOps
import torch
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="crest")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold"})

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"

if os.environ.get("NS_CAP"):
    CAP = Path(os.environ["NS_CAP"])
else:
    CAP = Path(glob.glob("/content/drive/MyDrive/Colab Notebooks/Training_Great Learning/MDS*/Capstone")[0])
ZIP  = Path(os.environ.get("NS_ZIP", CAP / "Food_101.zip"))
WORK = Path(os.environ.get("NS_WORK", "/content/work")); WORK.mkdir(parents=True, exist_ok=True)
ART  = CAP / "nutriscan_artifacts"; ART.mkdir(exist_ok=True)       # persistent outputs on Drive
FIG  = ART / "figures"; FIG.mkdir(exist_ok=True)
PROGRESS = ART / "progress.log"
def log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line)
    with open(PROGRESS, "a") as f: f.write(line + "\n")
def savefig(name):
    plt.savefig(FIG / f"{name}.png", bbox_inches="tight", dpi=130)

# budgets (kept small in QUICK smoke-test mode)
CLS_EPOCHS   = 2 if QUICK else 15
EFF_EPOCHS   = 1 if QUICK else 6
DET_EPOCHS   = 2 if QUICK else 30
DET_PER_CLASS = (6, 2, 2) if QUICK else (320, 60, 60)    # train / val / test images per class for detection
log(f"start | device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'} | quick={QUICK}")

## 1. Data loading and integrity checks
The zip holds one folder per food class. It also carries macOS metadata (`__MACOSX/`, `.DS_Store`), which we skip.
There are **no bounding-box annotations**, only class folders. Section 5 deals with that.

In [ ]:
DATA = WORK / "food"
if not DATA.exists():
    with zipfile.ZipFile(ZIP) as zf:
        members = [m for m in zf.namelist()
                   if not m.startswith("__MACOSX") and not m.endswith(".DS_Store") and m.lower().endswith((".jpg", ".jpeg", ".png"))]
        for m in members:
            parts = Path(m).parts
            dst = DATA / parts[-2] / parts[-1]
            dst.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(m) as src, open(dst, "wb") as out:
                shutil.copyfileobj(src, out)
CLASSES = sorted(p.name for p in DATA.iterdir() if p.is_dir())
print(len(CLASSES), "classes:", CLASSES)

In [ ]:
# Verify every image opens, record size / mode / brightness, hash for duplicate detection
rows = []
for c in CLASSES:
    for f in sorted((DATA / c).iterdir()):
        try:
            with Image.open(f) as im:
                im.verify()
            with Image.open(f) as im:
                w, h = im.size; mode = im.mode
                small = im.convert("RGB").resize((32, 32))
                arr = np.asarray(small, dtype=np.float32)
            rows.append(dict(path=str(f), label=c, w=w, h=h, mode=mode, ok=True,
                             r=arr[..., 0].mean(), g=arr[..., 1].mean(), b=arr[..., 2].mean(),
                             bright=arr.mean(), md5=hashlib.md5(f.read_bytes()).hexdigest()))
        except Exception as e:
            rows.append(dict(path=str(f), label=c, ok=False, err=str(e)))
df = pd.DataFrame(rows)
bad = df[~df.ok]
print(f"{len(df):,} files | corrupt: {len(bad)}")
df = df[df.ok].copy()
dups = df[df.duplicated("md5", keep=False)].sort_values("md5")
print("exact duplicate files:", len(dups), "| non-RGB modes:", df["mode"].value_counts().to_dict())
df = df.drop_duplicates("md5").reset_index(drop=True)
df["aspect"] = df.w / df.h
df.head()

## 2. Exploratory data analysis

In [ ]:
counts = df.label.value_counts().sort_values()
fig, ax = plt.subplots(figsize=(9, 5.5))
colors = ["#e4572e" if v < counts.median() * 0.5 else "#2f7d4f" for v in counts.values]
ax.barh([c.replace("_", " ") for c in counts.index], counts.values, color=colors)
for i, v in enumerate(counts.values): ax.text(v + 8, i, f"{v}", va="center", fontsize=9)
ax.set_title("Images per class"); ax.set_xlabel("images")
savefig("class_distribution"); plt.show()
imb = counts.max() / counts.min()
print(f"Imbalance ratio max/min = {imb:.1f}x  (smallest: {counts.idxmin()} = {counts.min()})")

**Reading the chart.** 16 classes are balanced at about 1,000 images each. `apple_pie` has roughly a quarter of that.
A model trained naively would see apple pie 4× less often and would likely under-predict it.
We handle this in training by oversampling apple pie (classification) and in evaluation by reporting **macro-F1**,
which weighs every class equally, next to accuracy.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df.w, bins=40, color="#2f7d4f", alpha=.8, label="width"); axes[0].hist(df.h, bins=40, color="#f2a541", alpha=.7, label="height")
axes[0].set_title("Image width / height (px)"); axes[0].legend()
axes[1].hist(df.aspect, bins=40, color="#3a86ff"); axes[1].set_title("Aspect ratio (w/h)")
sns.boxplot(data=df, y="label", x="bright", ax=axes[2], color="#52b788", fliersize=1)
axes[2].set_title("Mean brightness per class"); axes[2].set_ylabel("")
plt.tight_layout(); savefig("image_stats"); plt.show()
print(df[["w", "h", "aspect", "bright"]].describe().round(1))

In [ ]:
# A sample grid: 6 random images for each class
n = 6
fig, axes = plt.subplots(len(CLASSES), n, figsize=(n * 1.9, len(CLASSES) * 1.9))
for i, c in enumerate(CLASSES):
    sample = df[df.label == c].sample(min(n, (df.label == c).sum()), random_state=SEED)
    for j in range(n):
        ax = axes[i, j]; ax.axis("off")
        if j < len(sample):
            ax.imshow(ImageOps.fit(Image.open(sample.iloc[j].path).convert("RGB"), (160, 160)))
        if j == 0: ax.set_title(c.replace("_", " "), loc="left", fontsize=9)
plt.tight_layout(); savefig("sample_grid"); plt.show()

In [ ]:
# Average colour per class: hints at which classes the model could confuse on colour alone
col = df.groupby("label")[["r", "g", "b"]].mean()
fig, ax = plt.subplots(figsize=(10, 2.2))
ax.imshow(col.values[None, :, :] / 255, aspect="auto")
ax.set_xticks(range(len(col))); ax.set_xticklabels([c.replace("_", "\n") for c in col.index], fontsize=7); ax.set_yticks([])
ax.set_title("Mean RGB colour per class"); savefig("mean_colour"); plt.show()

**EDA takeaways**
- Images are web photos with varied size (mostly ≤ 512 px) and aspect ratio. We resize to a fixed input size, and letterboxing/cropping is handled by the training pipelines.
- Several fried dishes (French fries, onion rings, churro-like donuts, samosa, falafel, spring rolls) share warm golden-brown colours. Expect confusion among them.
- Dessert classes (chocolate cake, ice cream, strawberry shortcake, waffles, pancakes) differ more in texture and toppings than in colour.
- Most photos show a **single dominant dish**. That matters for detection: each image will usually yield one main box.

## 3. Stratified train / validation / test split (70 / 15 / 15)

In [ ]:
from sklearn.model_selection import train_test_split
tr, tmp = train_test_split(df, test_size=0.30, stratify=df.label, random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=tmp.label, random_state=SEED)
df["split"] = "train"; df.loc[va.index, "split"] = "val"; df.loc[te.index, "split"] = "test"
df[["path", "label", "split"]].to_csv(ART / "split.csv", index=False)
pd.crosstab(df.label, df.split)[["train", "val", "test"]].T

## 4. Classification
We compare two transfer-learning approaches, both pretrained on ImageNet:

| Model | Why |
|---|---|
| **YOLO11s-cls** (Ultralytics) | Fast, deploys with the same library as the detector, strong augmentation built in |
| **EfficientNet-B0** (torchvision) | Classic compound-scaled CNN baseline (Tan & Le, 2019), trained with a hand-written PyTorch loop |

**Class imbalance.** For YOLO-cls we oversample `apple_pie` in the training folder (×3 via symlinks).
For EfficientNet we use a `WeightedRandomSampler`. Both are evaluated on the untouched test split.

In [ ]:
CLS_DS = WORK / "cls_ds"
if not CLS_DS.exists():
    target = int(df[df.split == "train"].label.value_counts().median())
    for split in ["train", "val", "test"]:
        sub = df[df.split == split]
        for c in CLASSES:
            (CLS_DS / split / c).mkdir(parents=True, exist_ok=True)
            files = sub[sub.label == c].path.tolist()
            reps = max(1, round(target / max(len(files), 1))) if split == "train" else 1
            for r in range(reps):
                for f in files:
                    dst = CLS_DS / split / c / (f"r{r}_" + Path(f).name)
                    if not dst.exists(): os.symlink(f, dst)
for s in ["train", "val", "test"]:
    print(s, sum(len(os.listdir(CLS_DS / s / c)) for c in CLASSES))

In [ ]:
from ultralytics import YOLO
CLS_BEST = ART / "weights" / "classifier.pt"; CLS_BEST.parent.mkdir(exist_ok=True)
if not CLS_BEST.exists():
    log("train YOLO11s-cls")
    m = YOLO("yolo11s-cls.pt")
    r = m.train(data=str(CLS_DS), epochs=CLS_EPOCHS, imgsz=224, batch=64 if not QUICK else 16, device=DEVICE,
                project=str(WORK / "runs"), name="cls", exist_ok=True, seed=SEED, patience=5,
                optimizer="AdamW", lr0=1e-3, cos_lr=True, workers=2, plots=True, verbose=False)
    shutil.copy(Path(r.save_dir) / "weights" / "best.pt", CLS_BEST)
    shutil.copy(Path(r.save_dir) / "results.csv", ART / "cls_results.csv")
    log("done YOLO11s-cls")
hist = pd.read_csv(ART / "cls_results.csv"); hist.columns = hist.columns.str.strip()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(hist.epoch, hist["train/loss"], label="train"); ax[0].plot(hist.epoch, hist["val/loss"], label="val"); ax[0].set_title("YOLO11s-cls loss"); ax[0].legend()
ax[1].plot(hist.epoch, hist["metrics/accuracy_top1"], label="top-1"); ax[1].plot(hist.epoch, hist["metrics/accuracy_top5"], label="top-5"); ax[1].set_title("Validation accuracy"); ax[1].legend()
savefig("cls_curves"); plt.show()

In [ ]:
# EfficientNet-B0 baseline with a plain PyTorch training loop
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.amp import autocast, GradScaler

IDX = {c: i for i, c in enumerate(CLASSES)}
norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_train = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1)), T.RandomHorizontalFlip(), T.ColorJitter(.2, .2, .2), T.ToTensor(), norm])
tf_eval  = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), norm])
class DS(Dataset):
    def __init__(s, frame, tf): s.p, s.y, s.tf = frame.path.tolist(), frame.label.map(IDX).tolist(), tf
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return s.tf(Image.open(s.p[i]).convert("RGB")), s.y[i]

EFF_BEST = ART / "weights" / "efficientnet_b0.pt"
NW = 0 if QUICK else 2
trn = df[df.split == "train"]
w = trn.label.map(1.0 / trn.label.value_counts()).values
dl_tr = DataLoader(DS(trn, tf_train), batch_size=8 if QUICK else 64, sampler=WeightedRandomSampler(w, len(w), replacement=True), num_workers=NW)
dl_va = DataLoader(DS(df[df.split == "val"], tf_eval), batch_size=16 if QUICK else 128, num_workers=NW)
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:
    eff = efficientnet_b0(weights=None if QUICK else EfficientNet_B0_Weights.IMAGENET1K_V1)
except Exception as e:
    print("pretrained weights unavailable, training from scratch:", e); eff = efficientnet_b0(weights=None)
eff.classifier[1] = nn.Linear(eff.classifier[1].in_features, len(CLASSES))
eff = eff.to(dev)
eff_hist = []
if EFF_BEST.exists():
    eff.load_state_dict(torch.load(EFF_BEST, map_location=dev))
    eff_hist = json.load(open(ART / "eff_hist.json"))
else:
    log("train EfficientNet-B0")
    opt = torch.optim.AdamW(eff.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=EFF_EPOCHS * len(dl_tr))
    scaler = GradScaler(enabled=dev == "cuda"); lossf = nn.CrossEntropyLoss(label_smoothing=0.1); best = 0
    for ep in range(EFF_EPOCHS):
        eff.train(); tl = 0
        for i, (x, y) in enumerate(dl_tr):
            x, y = x.to(dev), y.to(dev); opt.zero_grad()
            with autocast(device_type="cuda", enabled=dev == "cuda"):
                loss = lossf(eff(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); tl += loss.item()
            if QUICK and i > 2: break
        eff.eval(); correct = n = 0
        with torch.no_grad(), autocast(device_type="cuda", enabled=dev == "cuda"):
            for x, y in dl_va:
                correct += (eff(x.to(dev)).argmax(1).cpu() == y).sum().item(); n += len(y)
        acc = correct / n; eff_hist.append(dict(epoch=ep + 1, train_loss=tl / len(dl_tr), val_acc=acc))
        print(eff_hist[-1])
        if acc >= best: best = acc; torch.save(eff.state_dict(), EFF_BEST)
    json.dump(eff_hist, open(ART / "eff_hist.json", "w"))
    eff.load_state_dict(torch.load(EFF_BEST, map_location=dev))
    log("done EfficientNet-B0")
pd.DataFrame(eff_hist)

In [ ]:
# Test-set evaluation for both classifiers
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
test = df[df.split == "test"].reset_index(drop=True)
y_true = test.label.map(IDX).values

yolo_cls = YOLO(str(CLS_BEST))
t0 = time.time(); y_yolo, p_yolo = [], []
for i in range(0, len(test), 64):
    for r in yolo_cls.predict(test.path[i:i + 64].tolist(), imgsz=224, device=DEVICE, verbose=False):
        y_yolo.append(IDX[r.names[int(r.probs.top1)]]); p_yolo.append(float(r.probs.top1conf))
ms_yolo = (time.time() - t0) / len(test) * 1000

dl_te = DataLoader(DS(test, tf_eval), batch_size=16 if QUICK else 128, num_workers=NW)
eff.eval(); y_eff = []; t0 = time.time()
with torch.no_grad():
    for x, _ in dl_te: y_eff += eff(x.to(dev)).argmax(1).cpu().tolist()
ms_eff = (time.time() - t0) / len(test) * 1000

cmp = pd.DataFrame({
    "model": ["YOLO11s-cls", "EfficientNet-B0"],
    "accuracy": [accuracy_score(y_true, y_yolo), accuracy_score(y_true, y_eff)],
    "macro_F1": [f1_score(y_true, y_yolo, average="macro"), f1_score(y_true, y_eff, average="macro")],
    "apple_pie_F1": [f1_score(y_true, y_yolo, labels=[IDX["apple_pie"]], average="macro"),
                     f1_score(y_true, y_eff, labels=[IDX["apple_pie"]], average="macro")],
    "ms_per_image": [ms_yolo, ms_eff],
    "params_M": [sum(p.numel() for p in yolo_cls.model.parameters()) / 1e6, sum(p.numel() for p in eff.parameters()) / 1e6],
}).round(4)
cmp.to_csv(ART / "classifier_comparison.csv", index=False)
cmp

In [ ]:
best_name = cmp.sort_values("macro_F1", ascending=False).iloc[0].model
y_best = y_yolo if best_name == "YOLO11s-cls" else y_eff
print("Best classifier on test macro-F1:", best_name, "\n")
rep = classification_report(y_true, y_best, target_names=CLASSES, digits=3, output_dict=True)
print(classification_report(y_true, y_best, target_names=CLASSES, digits=3))
cm = confusion_matrix(y_true, y_best, normalize="true")
fig, ax = plt.subplots(figsize=(10, 8.5))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Greens", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax, cbar=False, annot_kws={"size": 7})
ax.set_title(f"Normalised confusion matrix — {best_name} (test)"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
savefig("cls_confusion"); plt.show()

In [ ]:
# Most frequent confusions
pairs = pd.DataFrame({"true": [CLASSES[i] for i in y_true], "pred": [CLASSES[i] for i in y_best]})
top_conf = pairs[pairs.true != pairs.pred].value_counts().head(10).rename("count").reset_index()
top_conf

In [ ]:
# What the misclassified images look like
wrong = np.where(np.array(y_best) != y_true)[0]
pick = np.random.RandomState(SEED).choice(wrong, min(12, len(wrong)), replace=False) if len(wrong) else []
fig, axes = plt.subplots(2, 6, figsize=(15, 5.6))
for ax, i in zip(axes.flat, pick):
    ax.imshow(ImageOps.fit(Image.open(test.path[i]).convert("RGB"), (220, 220))); ax.axis("off")
    ax.set_title(f"true: {CLASSES[y_true[i]]}\npred: {CLASSES[y_best[i]]}", fontsize=8, color="#b23a1e")
for ax in list(axes.flat)[len(pick):]: ax.axis("off")
plt.suptitle("Sample classification errors", fontweight="bold"); plt.tight_layout(); savefig("cls_errors"); plt.show()

**Classification findings.** The comparison table, report and confusion matrix above are the evidence. The pattern to look for:
errors concentrate in visually similar pairs, usually fried golden items and layered desserts, while dishes with distinctive shape
(pizza, tacos, spring rolls) score highest. The best model is the one with the higher macro-F1. That model is exported for the app.

## 5. Object detection
### 5.1 Creating box labels
The dataset has class folders but **no bounding boxes**, so a detector cannot be trained directly. Hand-drawing ~5,000 boxes is
outside this project's scope. Instead we use **Grounding DINO** (Liu et al., 2023), an open-vocabulary detector that finds objects
from a text prompt. We prompt it with `"food."` and with the class name. We keep confident boxes and assign each box the folder's class.

This is **pseudo-labelling**. Its limits:
- Boxes come from a model, not a person, so detector mAP is measured against these pseudo-labels.
- Images where Grounding DINO finds nothing confident fall back to one box covering the dish area. They are counted and reported.
- Section 6 adds a check that does not depend on the pseudo-labels: does the detect-then-classify pipeline name the right dish?

In [ ]:
rs = np.random.RandomState(SEED)
det_rows = []
for split, k in zip(["train", "val", "test"], DET_PER_CLASS):
    sub = df[df.split == split]
    for c in CLASSES:
        s = sub[sub.label == c]
        det_rows.append(s.sample(min(k, len(s)), random_state=SEED))
det_df = pd.concat(det_rows).reset_index(drop=True)
print(det_df.groupby("split").size().to_dict())

In [ ]:
PSEUDO = ART / "pseudo_boxes.json"
boxes = json.load(open(PSEUDO)) if PSEUDO.exists() else {}
todo = [p for p in det_df.path if p not in boxes]
gd_ok = False
if todo and not QUICK:
    try:
        from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
        gd_id = "IDEA-Research/grounding-dino-tiny"
        proc = AutoProcessor.from_pretrained(gd_id)
        gd = AutoModelForZeroShotObjectDetection.from_pretrained(gd_id).to(dev).eval()
        gd_ok = True
    except Exception as e:
        print("Grounding DINO unavailable:", e)

def gd_detect(paths, labels):
    ims = [Image.open(p).convert("RGB") for p in paths]
    texts = [f"food. {l.replace('_', ' ')}." for l in labels]
    inp = proc(images=ims, text=texts, return_tensors="pt", padding=True).to(dev)
    with torch.no_grad():
        out = gd(**inp)
    sizes = [im.size[::-1] for im in ims]
    try:
        res = proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=0.30, text_threshold=0.25, target_sizes=sizes)
    except TypeError:
        res = proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=0.30, text_threshold=0.25, target_sizes=sizes)
    return ims, res

def fallback_box(w, h):   # central box covering the typical dish region
    return [0.1 * w, 0.1 * h, 0.9 * w, 0.9 * h]

def clean(bxs, scores, w, h, max_n=3):
    keep = []
    order = np.argsort(scores)[::-1]
    for i in order:
        b = bxs[i]; area = (b[2] - b[0]) * (b[3] - b[1]) / (w * h)
        if area < 0.02: continue                      # specks
        if area > 0.97 and len(bxs) > 1: continue     # whole-frame duplicates when better boxes exist
        if any(iou(b, k[0]) > 0.6 for k in keep): continue
        keep.append((b, float(scores[i])))
        if len(keep) == max_n: break
    return keep

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy; u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u else 0

lab_of = dict(zip(det_df.path, det_df.label))
log(f"pseudo-labelling {len(todo)} images (grounding dino={gd_ok})")
B = 16
for i in range(0, len(todo), B):
    chunk = todo[i:i + B]
    if gd_ok:
        ims, res = gd_detect(chunk, [lab_of[p] for p in chunk])
        for p, im, r in zip(chunk, ims, res):
            w, h = im.size
            kept = clean(r["boxes"].cpu().numpy().tolist(), r["scores"].cpu().numpy(), w, h)
            boxes[p] = dict(w=w, h=h, boxes=[k[0] for k in kept], scores=[k[1] for k in kept], src="gdino" if kept else "fallback")
            if not kept: boxes[p]["boxes"] = [fallback_box(w, h)]
    else:
        for p in chunk:
            w, h = Image.open(p).size
            boxes[p] = dict(w=w, h=h, boxes=[fallback_box(w, h)], scores=[1.0], src="fallback")
    if (i // B) % 50 == 0:
        json.dump(boxes, open(PSEUDO, "w")); print(f"{i + len(chunk)}/{len(todo)}")
json.dump(boxes, open(PSEUDO, "w"))
src = pd.Series([boxes[p]["src"] for p in det_df.path]).value_counts()
nbox = pd.Series([len(boxes[p]["boxes"]) for p in det_df.path]).value_counts().sort_index()
print("label source:", src.to_dict(), "| boxes per image:", nbox.to_dict())
log("pseudo-labelling done")

In [ ]:
# Quality check of pseudo-labels: area distribution and a visual sample
area = [ (b[2]-b[0])*(b[3]-b[1])/(boxes[p]["w"]*boxes[p]["h"]) for p in det_df.path for b in boxes[p]["boxes"]]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(area, bins=np.linspace(0, 1, 31), color="#2f7d4f"); ax[0].set_title("Pseudo-box area (fraction of image)")
pd.Series([boxes[p]["src"] for p in det_df.path]).value_counts().plot.bar(ax=ax[1], color=["#2f7d4f", "#e4572e"]); ax[1].set_title("Label source")
savefig("pseudo_stats"); plt.show()

from PIL import ImageDraw
smp = det_df.sample(12, random_state=1)
fig, axes = plt.subplots(2, 6, figsize=(16, 5.8))
for ax, (_, r) in zip(axes.flat, smp.iterrows()):
    im = Image.open(r.path).convert("RGB"); d = ImageDraw.Draw(im)
    for b in boxes[r.path]["boxes"]: d.rectangle(b, outline="#00e676", width=4)
    ax.imshow(im); ax.axis("off"); ax.set_title(f"{r.label} ({boxes[r.path]['src']})", fontsize=8)
plt.suptitle("Pseudo-labelled boxes (green)", fontweight="bold"); plt.tight_layout(); savefig("pseudo_samples"); plt.show()

In [ ]:
# Write the YOLO detection dataset
DET_DS = WORK / "det_ds"
for split in ["train", "val", "test"]:
    (DET_DS / "images" / split).mkdir(parents=True, exist_ok=True); (DET_DS / "labels" / split).mkdir(parents=True, exist_ok=True)
for _, r in det_df.iterrows():
    stem = f"{r.label}__{Path(r.path).stem}"
    dst = DET_DS / "images" / r.split / f"{stem}.jpg"
    if not dst.exists(): os.symlink(r.path, dst)
    bx = boxes[r.path]; W, H = bx["w"], bx["h"]
    with open(DET_DS / "labels" / r.split / f"{stem}.txt", "w") as f:
        for b in bx["boxes"]:
            cx, cy = (b[0] + b[2]) / 2 / W, (b[1] + b[3]) / 2 / H; bw, bh = (b[2] - b[0]) / W, (b[3] - b[1]) / H
            f.write(f"{IDX[r.label]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
yaml_path = DET_DS / "food.yaml"
yaml_path.write_text(f"path: {DET_DS}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n" +
                     "".join(f"  {i}: {c}\n" for i, c in enumerate(CLASSES)))
print(yaml_path.read_text()[:300])

### 5.2 Training the YOLO11s detector
YOLO11 (Ultralytics, 2024) is a single-stage, anchor-free detector. It predicts boxes and class scores in one pass, which suits
a phone app that needs answers in well under a second. We fine-tune the COCO-pretrained `yolo11s.pt` (9.4 M parameters).

In [ ]:
DET_BEST = ART / "weights" / "detector.pt"
if not DET_BEST.exists():
    log("train YOLO11s detector")
    det = YOLO("yolo11s.pt")
    r = det.train(data=str(yaml_path), epochs=DET_EPOCHS, imgsz=512, batch=32 if not QUICK else 8, device=DEVICE,
                  project=str(WORK / "runs"), name="det", exist_ok=True, seed=SEED, patience=10, cos_lr=True,
                  close_mosaic=5, workers=2, plots=True, verbose=False)
    shutil.copy(Path(r.save_dir) / "weights" / "best.pt", DET_BEST)
    shutil.copy(Path(r.save_dir) / "results.csv", ART / "det_results.csv")
    for f in ["results.png", "BoxPR_curve.png", "PR_curve.png", "confusion_matrix_normalized.png"]:
        if (Path(r.save_dir) / f).exists(): shutil.copy(Path(r.save_dir) / f, FIG / f"det_{f}")
    log("done detector")
dh = pd.read_csv(ART / "det_results.csv"); dh.columns = dh.columns.str.strip()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for k in ["train/box_loss", "val/box_loss", "train/cls_loss", "val/cls_loss"]: ax[0].plot(dh.epoch, dh[k], label=k)
ax[0].legend(fontsize=7); ax[0].set_title("Detector losses")
ax[1].plot(dh.epoch, dh["metrics/mAP50(B)"], label="mAP@0.5"); ax[1].plot(dh.epoch, dh["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")
ax[1].legend(); ax[1].set_title("Validation mAP"); savefig("det_curves"); plt.show()

In [ ]:
detector = YOLO(str(DET_BEST))
dm = detector.val(data=str(yaml_path), split="test", imgsz=512, device=DEVICE, verbose=False, plots=True,
                  project=str(WORK / "runs"), name="det_test", exist_ok=True)
det_metrics = {"mAP50": float(dm.box.map50), "mAP50_95": float(dm.box.map), "precision": float(dm.box.mp), "recall": float(dm.box.mr)}
print({k: round(v, 4) for k, v in det_metrics.items()})
per_cls = pd.DataFrame({"class": CLASSES, "AP50_95": [float(dm.box.maps[i]) for i in range(len(CLASSES))]}).sort_values("AP50_95")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(per_cls["class"].str.replace("_", " "), per_cls.AP50_95, color="#2f7d4f"); ax.set_title("Per-class AP@0.5:0.95 (test, vs pseudo-labels)")
savefig("det_per_class"); plt.show()

In [ ]:
# Detector predictions on unseen test images
smp = det_df[det_df.split == "test"].sample(12, random_state=3)
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, (_, r) in zip(axes.flat, smp.iterrows()):
    res = detector.predict(r.path, imgsz=512, conf=0.25, device=DEVICE, verbose=False)[0]
    ax.imshow(res.plot()[..., ::-1]); ax.axis("off"); ax.set_title(f"true: {r.label}", fontsize=8)
plt.suptitle("YOLO11s detections on test images", fontweight="bold"); plt.tight_layout(); savefig("det_samples"); plt.show()

## 6. Two-stage pipeline: detect → classify
In production, the detector proposes regions and the stronger classifier re-labels each crop. We compare three ways of naming the dish
in each test image (true label = folder):

- **A. Detector only**: label of the highest-confidence box.
- **B. Classifier on whole image**: no localisation.
- **C. Detect → classify**: classifier applied to the top box crop, keeping whichever of the two labels is more confident.

This check uses the real folder labels, so it does not depend on the pseudo-boxes.

In [ ]:
tst = det_df[det_df.split == "test"].reset_index(drop=True)
A, Bp, C, lat = [], [], [], []
for _, r in tst.iterrows():
    im = Image.open(r.path).convert("RGB"); t0 = time.time()
    d = detector.predict(im, imgsz=512, conf=0.1, device=DEVICE, verbose=False)[0]
    if len(d.boxes):
        j = int(d.boxes.conf.argmax()); dl, dc = d.names[int(d.boxes.cls[j])], float(d.boxes.conf[j])
        crop = im.crop(tuple(int(v) for v in d.boxes.xyxy[j].tolist()))
    else:
        dl, dc, crop = None, 0.0, im
    c = yolo_cls.predict(crop, imgsz=224, device=DEVICE, verbose=False)[0]
    cl, cc = c.names[int(c.probs.top1)], float(c.probs.top1conf)
    lat.append((time.time() - t0) * 1000)
    w = yolo_cls.predict(im, imgsz=224, device=DEVICE, verbose=False)[0]
    A.append(dl); Bp.append(w.names[int(w.probs.top1)]); C.append(cl if cc >= dc else dl)
pipe = pd.DataFrame({
    "approach": ["A. detector only", "B. classifier, whole image", "C. detect → classify"],
    "accuracy": [np.mean([a == t for a, t in zip(A, tst.label)]), np.mean([b == t for b, t in zip(Bp, tst.label)]),
                 np.mean([c == t for c, t in zip(C, tst.label)])]}).round(4)
pipe["note"] = ["", "", f"median latency {np.median(lat):.0f} ms/image"]
pipe.to_csv(ART / "pipeline_comparison.csv", index=False)
pipe

**Reading the pipeline table.** The detector alone has to localise and classify with fewer training images (it saw ~320 per class,
the classifier ~700). Re-labelling the crop with the classifier usually recovers accuracy while keeping the box. That box is what lets
the app handle **plates with several items** and estimate portion size from box area. The app ships approach C.

## 7. Nutrition mapping and meal analytics

In [ ]:
NUTRITION = json.loads(r'''{
 "apple_pie": {
  "serving_g": 125,
  "kcal": 296,
  "protein_g": 2.4,
  "carbs_g": 42.5,
  "fat_g": 13.8,
  "group": "Dessert",
  "swap": "Baked apple with cinnamon and Greek yogurt"
 },
 "chocolate_cake": {
  "serving_g": 110,
  "kcal": 420,
  "protein_g": 5,
  "carbs_g": 55,
  "fat_g": 21,
  "group": "Dessert",
  "swap": "Dark chocolate square + fruit"
 },
 "donuts": {
  "serving_g": 75,
  "kcal": 300,
  "protein_g": 4,
  "carbs_g": 34,
  "fat_g": 17,
  "group": "Dessert",
  "swap": "Whole-grain toast with nut butter"
 },
 "falafel": {
  "serving_g": 150,
  "kcal": 500,
  "protein_g": 20,
  "carbs_g": 48,
  "fat_g": 27,
  "group": "Middle Eastern",
  "swap": "Baked falafel"
 },
 "french_fries": {
  "serving_g": 150,
  "kcal": 470,
  "protein_g": 5,
  "carbs_g": 60,
  "fat_g": 23,
  "group": "Side",
  "swap": "Oven-baked potato or side salad"
 },
 "hot_dog": {
  "serving_g": 150,
  "kcal": 420,
  "protein_g": 15,
  "carbs_g": 32,
  "fat_g": 26,
  "group": "Fast food",
  "swap": "Chicken sausage on whole-wheat bun"
 },
 "ice_cream": {
  "serving_g": 130,
  "kcal": 270,
  "protein_g": 5,
  "carbs_g": 31,
  "fat_g": 14,
  "group": "Dessert",
  "swap": "Frozen banana nice-cream"
 },
 "nachos": {
  "serving_g": 250,
  "kcal": 690,
  "protein_g": 18,
  "carbs_g": 62,
  "fat_g": 40,
  "group": "Mexican",
  "swap": "Baked chips, beans, salsa, less cheese"
 },
 "onion_rings": {
  "serving_g": 150,
  "kcal": 480,
  "protein_g": 6,
  "carbs_g": 50,
  "fat_g": 28,
  "group": "Side",
  "swap": "Roasted onions or salad"
 },
 "pancakes": {
  "serving_g": 180,
  "kcal": 450,
  "protein_g": 10,
  "carbs_g": 62,
  "fat_g": 18,
  "group": "Breakfast",
  "swap": "Oat-banana pancakes, fruit topping"
 },
 "pizza": {
  "serving_g": 220,
  "kcal": 600,
  "protein_g": 25,
  "carbs_g": 70,
  "fat_g": 24,
  "group": "Fast food",
  "swap": "Thin crust, veg toppings, light cheese"
 },
 "ravioli": {
  "serving_g": 250,
  "kcal": 430,
  "protein_g": 18,
  "carbs_g": 52,
  "fat_g": 16,
  "group": "Pasta",
  "swap": "Tomato sauce instead of cream"
 },
 "samosa": {
  "serving_g": 100,
  "kcal": 300,
  "protein_g": 5,
  "carbs_g": 32,
  "fat_g": 17,
  "group": "Starter",
  "swap": "Baked samosa"
 },
 "spring_rolls": {
  "serving_g": 150,
  "kcal": 360,
  "protein_g": 8,
  "carbs_g": 40,
  "fat_g": 18,
  "group": "Asian",
  "swap": "Fresh (rice-paper) spring rolls"
 },
 "strawberry_shortcake": {
  "serving_g": 150,
  "kcal": 380,
  "protein_g": 5,
  "carbs_g": 52,
  "fat_g": 17,
  "group": "Dessert",
  "swap": "Strawberries with yogurt"
 },
 "tacos": {
  "serving_g": 200,
  "kcal": 420,
  "protein_g": 20,
  "carbs_g": 34,
  "fat_g": 22,
  "group": "Mexican",
  "swap": "Grilled fish tacos on corn tortillas"
 },
 "waffles": {
  "serving_g": 150,
  "kcal": 440,
  "protein_g": 10,
  "carbs_g": 56,
  "fat_g": 20,
  "group": "Breakfast",
  "swap": "Whole-grain waffle with fruit"
 }
}''')
nut = pd.DataFrame(NUTRITION).T
nut.index.name = "food"; nut[["kcal", "protein_g", "carbs_g", "fat_g"]] = nut[["kcal", "protein_g", "carbs_g", "fat_g"]].astype(float)
nut["kcal_per_100g"] = (nut.kcal / nut.serving_g.astype(float) * 100).round(0)
fig, ax = plt.subplots(figsize=(9, 5))
s = nut.sort_values("kcal")
ax.barh(s.index.str.replace("_", " "), s.protein_g * 4, color="#2f7d4f", label="protein")
ax.barh(s.index.str.replace("_", " "), s.carbs_g * 4, left=s.protein_g * 4, color="#f2a541", label="carbs")
ax.barh(s.index.str.replace("_", " "), s.fat_g * 9, left=s.protein_g * 4 + s.carbs_g * 4, color="#e4572e", label="fat")
ax.set_xlabel("kcal per typical serving"); ax.set_title("Calorie make-up of each dish"); ax.legend()
savefig("nutrition"); plt.show()
nut[["serving_g", "kcal", "protein_g", "carbs_g", "fat_g", "kcal_per_100g", "swap"]]

In [ ]:
def analyse_meal(path, portion=1.0):
    im = Image.open(path).convert("RGB")
    d = detector.predict(im, imgsz=512, conf=0.25, device=DEVICE, verbose=False)[0]
    items = []
    for b in d.boxes:
        box = [int(v) for v in b.xyxy[0].tolist()]; dl, dc = d.names[int(b.cls)], float(b.conf)
        c = yolo_cls.predict(im.crop(tuple(box)), imgsz=224, device=DEVICE, verbose=False)[0]
        cl, cc = c.names[int(c.probs.top1)], float(c.probs.top1conf)
        lab = cl if cc >= dc else dl
        n = NUTRITION[lab]
        items.append(dict(food=lab, confidence=round(max(cc, dc), 3), kcal=n["kcal"] * portion, protein_g=n["protein_g"] * portion,
                          carbs_g=n["carbs_g"] * portion, fat_g=n["fat_g"] * portion, healthier_swap=n["swap"]))
    return d, pd.DataFrame(items)

demo = tst.sample(3, random_state=7)
for _, r in demo.iterrows():
    d, items = analyse_meal(r.path)
    plt.figure(figsize=(4, 4)); plt.imshow(d.plot()[..., ::-1]); plt.axis("off"); plt.title(f"true: {r.label}"); plt.show()
    display(items)

In [ ]:
# Pattern analysis on a simulated 30-day food log (illustrates the "analyse user choices" objective)
rng = np.random.RandomState(SEED)
prefs = rng.dirichlet(np.ones(len(CLASSES)) * 0.4)               # one user's taste profile
log_rows = []
for day in pd.date_range("2026-10-01", periods=30):
    for meal in ["breakfast", "lunch", "dinner"] + (["snack"] if rng.rand() < .6 else []):
        f = rng.choice(CLASSES, p=prefs); n = NUTRITION[f]; portion = rng.choice([0.75, 1, 1.25], p=[.2, .6, .2])
        log_rows.append(dict(date=day, meal=meal, food=f, kcal=n["kcal"] * portion, protein_g=n["protein_g"] * portion,
                             carbs_g=n["carbs_g"] * portion, fat_g=n["fat_g"] * portion))
ulog = pd.DataFrame(log_rows)
daily = ulog.groupby("date")[["kcal", "protein_g", "carbs_g", "fat_g"]].sum()
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
ax[0].plot(daily.index, daily.kcal, color="#2f7d4f", marker="o", ms=3); ax[0].axhline(2000, ls="--", color="#e4572e", label="2,000 kcal goal")
ax[0].set_title("Daily calories (simulated user)"); ax[0].legend()
ulog.food.value_counts().head(8).plot.barh(ax=ax[1], color="#52b788"); ax[1].invert_yaxis(); ax[1].set_title("Most-logged foods")
plt.tight_layout(); savefig("user_log"); plt.show()
over = (daily.kcal > 2000).mean()
top = ulog.food.value_counts().index[:3]
print(f"Days over goal: {over:.0%}. Average intake: {daily.kcal.mean():.0f} kcal/day.")
print("Personalised suggestions:")
for f in top: print(f"  • You often log {f.replace('_', ' ')} → try: {NUTRITION[f]['swap']}")

## 8. Business insights, limitations and next steps
**What the models deliver for NutriScan**
- The classifier recognises the 17 dishes with the accuracy and macro-F1 shown in Section 4. Errors cluster in look-alike pairs, which
  tells the product team where a "did you mean …?" confirmation prompt adds the most value.
- The detector localises the dish so the app can handle more than one item per photo and scale portions by box size.
- Combining detector and classifier (Section 6) gives the best naming accuracy with latency suitable for a mobile backend.
- Mapping labels to a nutrition table turns a photo into calories and macros in one step. Logging those results over time supports
  the brief's goal of spotting patterns and suggesting healthier swaps.

**Limitations**
- **Detection labels are machine-generated.** Detector mAP is measured against Grounding DINO boxes, not hand-drawn ground truth.
  A few hundred hand-checked boxes would give a trustworthy mAP and could be used to fine-tune.
- **17 classes only.** Real meals contain hundreds of foods. Anything outside these classes gets forced into one of them.
  An "unknown food" threshold is needed before launch.
- **Nutrition is per typical serving** from public reference values. Portion estimation from a single 2-D photo is approximate.
  Depth sensing or a reference object (plate, card) would tighten it.
- **Apple pie is under-represented** (257 images vs ~1,000). Oversampling helps but more real images would help more.
- The user-pattern analysis uses a **simulated** log, to show the mechanism.

**Next steps**
1. Hand-verify ~500 pseudo-boxes with Label Studio or Roboflow, re-train, and report mAP on that clean set.
2. Extend to the full Food-101 (101 classes) and add multi-item plate photos (e.g. UEC-Food-256).
3. Add an open-set rejection threshold and a user-correction loop that feeds labelled data back into training.
4. Integrate with Apple Health / Google Fit through their APIs for longitudinal tracking.

**References**
- Bossard, Guillaumin & Van Gool (2014). *Food-101 — Mining Discriminative Components with Random Forests.* ECCV.
- Tan & Le (2019). *EfficientNet: Rethinking Model Scaling for CNNs.* ICML.
- Liu et al. (2023). *Grounding DINO: Marrying DINO with Grounded Pre-Training for Open-Set Object Detection.* arXiv:2303.05499.
- Jocher, Qiu & Chaurasia (2024). *Ultralytics YOLO11.* github.com/ultralytics/ultralytics.

## 9. Export artefacts and launch the Streamlit app

In [ ]:
metrics = {
    "Classifier": best_name,
    "Classifier test accuracy": f"{cmp.set_index('model').loc[best_name, 'accuracy']:.1%}",
    "Classifier macro-F1": f"{cmp.set_index('model').loc[best_name, 'macro_F1']:.3f}",
    "Detector mAP@0.5": f"{det_metrics['mAP50']:.3f}",
    "Detector mAP@0.5:0.95": f"{det_metrics['mAP50_95']:.3f}",
    "Pipeline accuracy (detect→classify)": f"{pipe.accuracy.iloc[2]:.1%}",
    "Classes": len(CLASSES),
}
json.dump(metrics, open(ART / "metrics.json", "w"), indent=1)
APP_DIR = ART / "app"; (APP_DIR / "weights").mkdir(parents=True, exist_ok=True); (APP_DIR / "samples").mkdir(exist_ok=True)
shutil.copy(DET_BEST, APP_DIR / "weights" / "detector.pt")
if best_name == "YOLO11s-cls":
    shutil.copy(CLS_BEST, APP_DIR / "weights" / "classifier.pt")
else:   # the app runs Ultralytics models; keep YOLO-cls as the in-app refiner and report EfficientNet in the notebook
    shutil.copy(CLS_BEST, APP_DIR / "weights" / "classifier.pt")
json.dump(NUTRITION, open(APP_DIR / "nutrition.json", "w"), indent=1)
json.dump(metrics, open(APP_DIR / "metrics.json", "w"), indent=1)
for c in CLASSES:
    p = tst[tst.label == c].path.iloc[0]
    ImageOps.contain(Image.open(p).convert("RGB"), (640, 640)).save(APP_DIR / "samples" / f"{c}__1.jpg", quality=88)
(APP_DIR / "requirements.txt").write_text("streamlit>=1.50\nultralytics>=8.3\nplotly\npandas\npillow\nhuggingface_hub\nopencv-python-headless\n")
log("artefacts exported")
metrics

In [ ]:
%%writefile {APP_DIR}/app.py
"""NutriScan - AI food detection & nutrition analysis (Streamlit).

Pipeline: YOLO detector finds food regions -> YOLO classifier labels each crop
(falls back to the detector's own label) -> nutrition lookup -> meal summary.
"""
import io, json, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
from PIL import Image, ImageDraw, ImageFont

ROOT = Path(__file__).parent
DET_W = ROOT / "weights" / "detector.pt"
CLS_W = ROOT / "weights" / "classifier.pt"
NUTRI = json.load(open(ROOT / "nutrition.json"))
METRICS = json.load(open(ROOT / "metrics.json")) if (ROOT / "metrics.json").exists() else {}

st.set_page_config(page_title="NutriScan · AI Food Detection", page_icon="🥗", layout="wide",
                   initial_sidebar_state="expanded")

# ----------------------------------------------------------------- styling
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600;9..144,800&family=Inter:wght@400;500;600&display=swap');
:root{--ink:#1d2a22;--muted:#5d6b62;--leaf:#2f7d4f;--leaf2:#52b788;--cream:#fbf8f1;--tomato:#e4572e;--card:#ffffff;}
html,body,[class*="css"]{font-family:'Inter',sans-serif;}
.stApp{background:radial-gradient(1200px 500px at 10% -10%,#e8f5ec 0%,transparent 60%),
       radial-gradient(900px 400px at 110% 0%,#fdeee6 0%,transparent 55%),var(--cream);}
.block-container{padding-top:1.4rem;max-width:1250px;}
h1,h2,h3{font-family:'Fraunces',serif!important;color:var(--ink);letter-spacing:-.01em;}
.hero{display:flex;justify-content:space-between;align-items:flex-end;gap:2rem;padding:1.6rem 2rem;border-radius:22px;
      background:linear-gradient(120deg,#16432c 0%,#2f7d4f 55%,#52b788 100%);color:#fff;box-shadow:0 18px 40px -18px #16432c88;margin-bottom:1.2rem;}
.hero h1{color:#fff!important;font-size:2.6rem;margin:0;line-height:1.05;}
.hero p{margin:.4rem 0 0;opacity:.88;font-size:1.02rem;max-width:560px;}
.pill{display:inline-block;padding:.25rem .7rem;border-radius:999px;background:#ffffff22;border:1px solid #ffffff44;font-size:.78rem;margin-right:.35rem;}
.card{background:var(--card);border-radius:18px;padding:1.1rem 1.2rem;border:1px solid #e9e4d8;box-shadow:0 8px 24px -16px #1d2a2240;}
.kpi{font-family:'Fraunces',serif;font-size:2.1rem;font-weight:800;color:var(--ink);line-height:1;}
.kpi-l{color:var(--muted);font-size:.8rem;text-transform:uppercase;letter-spacing:.08em;margin-top:.35rem;}
.item{display:flex;gap:1rem;align-items:center;background:#fff;border:1px solid #e9e4d8;border-radius:16px;padding:.75rem;margin-bottom:.7rem;}
.item img{width:84px;height:84px;object-fit:cover;border-radius:12px;}
.item .nm{font-family:'Fraunces',serif;font-weight:600;font-size:1.12rem;color:var(--ink);}
.tag{font-size:.72rem;padding:.12rem .5rem;border-radius:999px;background:#e8f5ec;color:var(--leaf);margin-left:.4rem;}
.macro{font-size:.85rem;color:var(--muted);}
.swap{font-size:.85rem;color:#8a3b1f;background:#fdeee6;border-radius:10px;padding:.35rem .6rem;margin-top:.35rem;display:inline-block;}
.conf{height:6px;border-radius:6px;background:#eee;overflow:hidden;margin-top:.4rem;width:180px;}
.conf>div{height:100%;background:linear-gradient(90deg,var(--leaf2),var(--leaf));}
section[data-testid="stSidebar"]{background:#16432c;}
section[data-testid="stSidebar"] *{color:#eaf4ee!important;}
.stButton>button,.stDownloadButton>button{border-radius:12px;border:0;background:var(--leaf);color:#fff;font-weight:600;}
.stTabs [data-baseweb="tab"]{font-weight:600;}
footer{visibility:hidden;}
</style>""", unsafe_allow_html=True)


# ----------------------------------------------------------------- models
@st.cache_resource(show_spinner="Loading models…")
def load_models():
    from ultralytics import YOLO
    repo = os.environ.get("NUTRISCAN_MODEL_REPO", "DatariusAI/nutriscan-food-models")
    paths = {}
    for key, local in (("det", DET_W), ("cls", CLS_W)):
        if local.exists():
            paths[key] = str(local)
        else:  # e.g. Streamlit Cloud: pull weights from the Hugging Face model repo
            try:
                from huggingface_hub import hf_hub_download
                paths[key] = hf_hub_download(repo, f"weights/{local.name}")
            except Exception:
                paths[key] = None
    det = YOLO(paths["det"]) if paths["det"] else None
    cls = YOLO(paths["cls"]) if paths["cls"] else None
    return det, cls


def pretty(name: str) -> str:
    return name.replace("_", " ").title()


PALETTE = ["#2f7d4f", "#e4572e", "#f2a541", "#3a86ff", "#8338ec", "#ff006e", "#06a77d", "#ffbe0b"]


def analyse(img: Image.Image, conf: float, use_cls: bool, portion: float):
    det, cls = load_models()
    rgb = img.convert("RGB")
    t0 = time.time()
    items = []
    if det is not None:
        r = det.predict(rgb, conf=conf, iou=0.5, verbose=False, imgsz=640)[0]
        for b in r.boxes:
            x1, y1, x2, y2 = [int(v) for v in b.xyxy[0].tolist()]
            label, score = r.names[int(b.cls)], float(b.conf)
            crop = rgb.crop((x1, y1, x2, y2))
            src = "detector"
            if use_cls and cls is not None and crop.width > 8 and crop.height > 8:
                c = cls.predict(crop, verbose=False, imgsz=224)[0]
                cl, cs = c.names[int(c.probs.top1)], float(c.probs.top1conf)
                if cs >= score:
                    label, score, src = cl, cs, "classifier"
            items.append(dict(label=label, conf=score, box=(x1, y1, x2, y2), crop=crop, src=src))
    if not items and cls is not None:  # no box found -> whole-image classification
        c = cls.predict(rgb, verbose=False, imgsz=224)[0]
        items.append(dict(label=c.names[int(c.probs.top1)], conf=float(c.probs.top1conf),
                          box=(0, 0, rgb.width, rgb.height), crop=rgb, src="classifier (whole image)"))
    ms = (time.time() - t0) * 1000

    # draw
    out = rgb.copy()
    d = ImageDraw.Draw(out)
    lw = max(3, rgb.width // 220)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", max(14, rgb.width // 45))
    except Exception:
        font = ImageFont.load_default()
    for i, it in enumerate(items):
        col = PALETTE[i % len(PALETTE)]
        x1, y1, x2, y2 = it["box"]
        d.rounded_rectangle((x1, y1, x2, y2), radius=lw * 3, outline=col, width=lw)
        txt = f" {pretty(it['label'])} {it['conf']:.0%} "
        tb = d.textbbox((x1, y1), txt, font=font)
        h = tb[3] - tb[1] + 8
        ty = max(0, y1 - h)
        d.rounded_rectangle((x1, ty, x1 + tb[2] - tb[0] + 6, ty + h), radius=6, fill=col)
        d.text((x1 + 3, ty + 2), txt, fill="white", font=font)
        n = NUTRI.get(it["label"], dict(serving_g=150, kcal=300, protein_g=10, carbs_g=35, fat_g=12,
                                        group="Food", swap="—"))
        it.update({k: (v * portion if isinstance(v, (int, float)) else v) for k, v in n.items()})
    return out, items, ms


def donut(p, c, f):
    kc = [p * 4, c * 4, f * 9]
    fig = go.Figure(go.Pie(labels=["Protein", "Carbs", "Fat"], values=kc, hole=.68, sort=False,
                           marker=dict(colors=["#2f7d4f", "#f2a541", "#e4572e"]), textinfo="percent"))
    fig.update_layout(height=260, margin=dict(t=10, b=10, l=10, r=10), showlegend=True,
                      legend=dict(orientation="h", y=-.08), paper_bgcolor="rgba(0,0,0,0)",
                      annotations=[dict(text=f"<b>{sum(kc):.0f}</b><br>kcal", showarrow=False, font_size=18)])
    return fig


def gauge(val, goal):
    fig = go.Figure(go.Indicator(mode="gauge+number", value=val, number=dict(suffix=" kcal"),
        gauge=dict(axis=dict(range=[0, goal]), bar=dict(color="#2f7d4f"),
                   steps=[dict(range=[0, goal * .8], color="#e8f5ec"), dict(range=[goal * .8, goal], color="#fdeee6")])))
    fig.update_layout(height=230, margin=dict(t=30, b=0, l=20, r=20), paper_bgcolor="rgba(0,0,0,0)")
    return fig


def img_b64(im: Image.Image) -> str:
    import base64
    im = im.copy(); im.thumbnail((200, 200))
    buf = io.BytesIO(); im.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode()


# ----------------------------------------------------------------- sidebar
with st.sidebar:
    st.markdown("## 🥗 NutriScan")
    st.caption("Snap a meal. See what's on the plate.")
    goal = st.number_input("Daily calorie goal", 1200, 4000, 2000, 50)
    portion = st.select_slider("Portion size", options=[0.5, 0.75, 1.0, 1.25, 1.5, 2.0], value=1.0,
                               format_func=lambda x: f"{x}× serving")
    conf = st.slider("Detection confidence", 0.05, 0.9, 0.25, 0.05)
    use_cls = st.toggle("Refine labels with classifier", value=True)
    st.divider()
    det, cls = load_models()
    st.markdown(f"**Detector:** {'✅ loaded' if det else '⚠️ missing'}  \n**Classifier:** {'✅ loaded' if cls else '⚠️ missing'}")
    if METRICS:
        st.markdown("**Model scorecard**")
        for k, v in METRICS.items():
            st.markdown(f"- {k}: **{v}**")

if "log" not in st.session_state:
    st.session_state.log = []

# ----------------------------------------------------------------- hero
st.markdown(f"""<div class="hero"><div>
<span class="pill">YOLO detection</span><span class="pill">Food classification</span><span class="pill">{len(NUTRI)} dishes</span>
<h1>Know your plate<br>in one photo.</h1>
<p>NutriScan finds every food item in your picture, names it, and turns it into calories and macros — with a healthier swap for each.</p>
</div><div style="font-size:4.2rem;line-height:1">🍱</div></div>""", unsafe_allow_html=True)

tab_scan, tab_log, tab_about = st.tabs(["📸  Scan a meal", "📒  Today's log", "ℹ️  How it works"])

with tab_scan:
    c1, c2 = st.columns([1, 1], gap="large")
    with c1:
        src = st.radio("Image source", ["Upload", "Camera", "Sample"], horizontal=True, label_visibility="collapsed")
        img = None
        if src == "Upload":
            f = st.file_uploader("Drop a food photo", type=["jpg", "jpeg", "png", "webp"])
            if f: img = Image.open(f)
        elif src == "Camera":
            f = st.camera_input("Take a photo")
            if f: img = Image.open(f)
        else:
            samples = sorted((ROOT / "samples").glob("*.jpg")) if (ROOT / "samples").exists() else []
            if samples:
                pick = st.selectbox("Sample image", samples, format_func=lambda p: pretty(p.stem.split("__")[0]))
                img = Image.open(pick)
            else:
                st.info("No sample images bundled.")
    if img is not None:
        with st.spinner("Scanning your plate…"):
            annotated, items, ms = analyse(img, conf, use_cls, portion)
        with c1:
            st.image(annotated, width="stretch")
            st.caption(f"{len(items)} item(s) · {ms:.0f} ms")
        with c2:
            if not items:
                st.warning("No food found. Try lowering the confidence threshold.")
            else:
                tk = sum(i["kcal"] for i in items); tp = sum(i["protein_g"] for i in items)
                tc = sum(i["carbs_g"] for i in items); tf = sum(i["fat_g"] for i in items)
                k1, k2, k3, k4 = st.columns(4)
                for col, v, l in [(k1, f"{tk:.0f}", "kcal"), (k2, f"{tp:.0f}g", "protein"),
                                  (k3, f"{tc:.0f}g", "carbs"), (k4, f"{tf:.0f}g", "fat")]:
                    col.markdown(f'<div class="card"><div class="kpi">{v}</div><div class="kpi-l">{l}</div></div>',
                                 unsafe_allow_html=True)
                st.plotly_chart(donut(tp, tc, tf), width="stretch", config={"displayModeBar": False})
                for it in items:
                    st.markdown(f"""<div class="item"><img src="data:image/jpeg;base64,{img_b64(it['crop'])}"/>
<div><div class="nm">{pretty(it['label'])}<span class="tag">{it['group']}</span></div>
<div class="macro">{it['kcal']:.0f} kcal · P {it['protein_g']:.0f}g · C {it['carbs_g']:.0f}g · F {it['fat_g']:.0f}g · ~{it['serving_g']:.0f} g</div>
<div class="conf"><div style="width:{it['conf']*100:.0f}%"></div></div>
<div class="swap">💡 Swap: {it['swap']}</div></div></div>""", unsafe_allow_html=True)
                if st.button("➕ Add this meal to today's log", width="stretch"):
                    for it in items:
                        st.session_state.log.append(dict(time=time.strftime("%H:%M"), food=pretty(it["label"]),
                            kcal=round(it["kcal"]), protein_g=round(it["protein_g"]),
                            carbs_g=round(it["carbs_g"]), fat_g=round(it["fat_g"])))
                    st.success("Logged!")
    else:
        with c2:
            st.markdown('<div class="card"><h3>How to use</h3><p>1. Upload or snap a meal photo.<br>'
                        '2. NutriScan boxes each food item and names it.<br>3. Review calories, macros and swaps.<br>'
                        '4. Add the meal to your daily log.</p></div>', unsafe_allow_html=True)

with tab_log:
    log = pd.DataFrame(st.session_state.log)
    if log.empty:
        st.info("Nothing logged yet. Scan a meal and press **Add to today's log**.")
    else:
        a, b = st.columns([1, 1.3], gap="large")
        with a:
            st.plotly_chart(gauge(log.kcal.sum(), goal), width="stretch", config={"displayModeBar": False})
            left = goal - log.kcal.sum()
            st.markdown(f"**{abs(left):.0f} kcal {'left' if left >= 0 else 'over'}** your goal of {goal}.")
        with b:
            st.dataframe(log, width="stretch", hide_index=True)
            st.download_button("⬇️ Export log (CSV)", log.to_csv(index=False), "nutriscan_log.csv", "text/csv")
        if st.button("Clear log"):
            st.session_state.log = []; st.rerun()

with tab_about:
    st.markdown("""
### Two-stage pipeline
1. **Detection** — a YOLO model fine-tuned on the food images finds each item and draws a box around it.
2. **Classification** — each box is cropped and passed to a YOLO classification model fine-tuned on the same classes.
   The app keeps whichever label has higher confidence.
3. **Nutrition** — the label is looked up in a per-serving nutrition table and scaled by the portion slider.

Nutrition values are approximate per typical serving and are for guidance only, not medical advice.
""")
    if METRICS:
        st.json(METRICS)


In [ ]:
# Launch the Streamlit app from this notebook and expose it with a Cloudflare quick tunnel
import re, stat, urllib.request
if not QUICK:
    shutil.copy(APP_DIR / "app.py", "/content/app.py") if (APP_DIR / "app.py").exists() else None
    for sub in ["weights", "samples"]:
        shutil.copytree(APP_DIR / sub, f"/content/{sub}", dirs_exist_ok=True)
    for f in ["nutrition.json", "metrics.json"]: shutil.copy(APP_DIR / f, f"/content/{f}")
    cf = "/content/cloudflared"
    if not os.path.exists(cf):
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf)
        os.chmod(cf, os.stat(cf).st_mode | stat.S_IEXEC)
    subprocess.Popen("cd /content && setsid nohup streamlit run app.py --server.port 8501 --server.headless true > /content/st.log 2>&1 &", shell=True)
    subprocess.Popen(f"setsid nohup {cf} tunnel --url http://localhost:8501 --no-autoupdate > /content/cf.log 2>&1 &", shell=True)
    url = None
    for _ in range(40):
        time.sleep(2)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cf.log").read() if os.path.exists("/content/cf.log") else "")
        if m: url = m.group(0); break
    print("NutriScan is live at:", url)
    (ART / "colab_app_url.txt").write_text(str(url))
    log(f"streamlit live {url}")